# Nehalennia data layer

In [1]:
# %pip install pandas openpyxl rdflib

from pathlib import Path

import pandas as pd

import collectie_zeeland
import rmo

OUT_DIR = Path("out")
OUT_DIR.mkdir(exist_ok=True)

EXCEL_CELL_LIMIT = 32767
LINK_COLUMNS = ["pid", "image_url", "edcs_pid", "iiif_manifest", "afbeelding_url"]


def write_excel(path, sheets):
    """One sheet per {name: frame}, with URL columns turned into clickable links."""
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for name, frame in sheets.items():
            # Excel rejects anything longer than 32767 characters in a single cell
            trimmed = frame.map(lambda v: v[:EXCEL_CELL_LIMIT] if isinstance(v, str) else v)
            trimmed.to_excel(writer, sheet_name=name, index=False)

        for sheet in writer.sheets.values():
            sheet.freeze_panes = "A2"
            sheet.auto_filter.ref = sheet.dimensions
            header = [c.value for c in sheet[1]]
            for column in LINK_COLUMNS:
                if column not in header:
                    continue
                index = header.index(column) + 1
                for (cell,) in sheet.iter_rows(min_row=2, min_col=index, max_col=index):
                    if cell.value:
                        cell.hyperlink = cell.value
                        cell.style = "Hyperlink"
    print(f"wrote {path.resolve()}")

## RMO

First run reads ~400 object pages at one request every half second, so allow a few
minutes; after that everything comes from `out/raw/` and `out/pages/`.

`rmo.fetch` then matches the transcriptions against the hand-made
`enrichment/edcs_nehalennia_analyse.xlsx`, adding `edcs_id`, `edcs_pid` and the
`edcs_score` the match was accepted on.

In [2]:
df_rmo, df_rmo_images = rmo.fetch(cache_dir=OUT_DIR)

write_excel(OUT_DIR / "rmo_nehalennia_colijnsplaat.xlsx",
            {"records": df_rmo, "images": df_rmo_images})
df_rmo.head()

nehalennia: 276 records
colijnsplaat: 386 records
411 unique records
reading 411 object pages for verwerving, cached ones are reused
    25/411     0s elapsed, ~0s to go
    50/411     0s elapsed, ~0s to go
    75/411     0s elapsed, ~0s to go
   100/411     0s elapsed, ~0s to go
   125/411     0s elapsed, ~0s to go
   150/411     0s elapsed, ~0s to go
   175/411     0s elapsed, ~0s to go
   200/411     0s elapsed, ~0s to go
   225/411     0s elapsed, ~0s to go
   250/411     0s elapsed, ~0s to go
   275/411     0s elapsed, ~0s to go
   300/411     0s elapsed, ~0s to go
   325/411     0s elapsed, ~0s to go
   350/411     0s elapsed, ~0s to go
   375/411     0s elapsed, ~0s to go
   400/411     0s elapsed, ~0s to go
   411/411     0s elapsed, ~0s to go
111 of 159 inscriptions matched to 77 of 137 EDCS entries
wrote /Users/Minh.Tran/Documents/rijks/cit_hackalod/2026-nehalennia/data_layer/out/rmo_nehalennia_colijnsplaat.xlsx


,pid,object,periode,datum_van,datum_tot,vindplaats,inscripties,beschrijving,bron,image_url,verwerving,gevonden_datum_van,gevonden_datum_tot,edcs_id,edcs_pid,edcs_score
0,https://hdl.handle.net/21.12126/137816,altaar,Romeins 150-250 n.Chr.,150,250,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",DE[AE] NEHALENNI[AE] / [.] VAL MAR[......] / N...,Altaar met nis. Vooraan rechtsonder is een hoe...,Rijksmuseum van Oudheden,https://www.rmo.nl/imageproxy/tiff/019732,opgraving RMO 1970,1970,1970,EDCS-09401505,https://edcs.hist.uzh.ch/monument/09401505,0.872
1,https://hdl.handle.net/21.12126/137817,altaar,Romeins 150-250 n.Chr.,150,250,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",DEAE . NEHALEN / NIAE . C . IVLIVS / PRIMITIVS...,"Altaar met nis. Kalksteen. Nehalennia zittend,...",Rijksmuseum van Oudheden,https://www.rmo.nl/imageproxy/tiff/019527,opgraving RMO 1970,1970,1970,EDCS-09401493,https://edcs.hist.uzh.ch/monument/09401493,1.000
2,https://hdl.handle.net/21.12126/137818,altaar,Romeins 150-250 n.Chr.,150,250,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",D . NEHALE / [N]IAE . PRIMAN / VS VRSVL[I] / P...,Altaar. Kalksteen. Rechter- en linkerkant: boo...,Rijksmuseum van Oudheden,https://www.rmo.nl/imageproxy/tiff/019423,opgraving RMO 1970,1970,1970,EDCS-09401502,https://edcs.hist.uzh.ch/monument/09401502,0.941
3,https://hdl.handle.net/21.12126/137819,altaar,Romeins 150-250 n.Chr.,150,250,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",NEHA[L]ENI[.] / [.]IMA[.....] / [---] / V [---],Altaar met nis. Kalksteen. Afwijkend type in z...,Rijksmuseum van Oudheden,https://www.rmo.nl/imageproxy/tiff/015782,opgraving RMO 1970,1970,1970,EDCS-23400585,https://edcs.hist.uzh.ch/monument/23400585,0.828
4,https://hdl.handle.net/21.12126/138005,altaar,Romeins 150-250 n.Chr.,150,250,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",DEAE / NEHALENIAE/ M . EXGINGIVS / AGRICOLA / ...,"Altaar met nis, kalksteen. Nehalennia zittend....",Rijksmuseum van Oudheden,https://www.rmo.nl/imageproxy/tiff/019737,opgraving RMO 1970,1970,1970,EDCS-09401487,https://edcs.hist.uzh.ch/monument/09401487,1.000


## Collectie Zeeland

Streams the dump on every run (no cache), which takes about 5 seconds.

In [ ]:
df_zeeland = collectie_zeeland.fetch(ttl_path=OUT_DIR / "zeeland_nehalennia.ttl")

write_excel(OUT_DIR / "zeeland_nehalennia.xlsx", {"objects": df_zeeland})
df_zeeland[["objectnummer", "titel", "objecttype", "materiaal", "vervaardiger"]]